# CSC 4792 Group 40: Kafue Town Council

This notebook documents the collection, cleaning, validation, and export of a pipe-delimited dataset from the official Kafue Town Council website.

The assignment requires CSV output, `|` as the separator, documented scraping/cleaning, and source traceability.

## 1. Configuration and source scope

We use the official council website only. The workflow covers the district profile, Quick Facts, council departments, mandate, CDF information, publications, and official news pages. Linked documents are captured as resource metadata and can be downloaded and parsed in the enrichment phase.

In [ ]:
from pathlib import Path
import pandas as pd

BASE_URL = 'https://www.kafuecouncil.gov.zm/'
DATA_PATH = Path('db-unza26-csc4792-kafue_council_information.csv')
df = pd.read_csv(DATA_PATH, sep='|', dtype=str).fillna('')
df.head()

## 2. Dataset structure

Each row is an observable fact, official news item, or linked council resource. `source_url` identifies the page where the record was observed; `retrieved_at` records the collection date.

In [ ]:
df.shape, df['record_type'].value_counts().to_dict(), df['category'].value_counts().to_dict()

In [ ]:
# Check assignment formatting requirements
assert DATA_PATH.name.startswith('db-unza26-csc4792-')
assert DATA_PATH.suffix == '.csv'
assert len(pd.read_csv(DATA_PATH, sep='|')) == len(df)
assert set(['record_type','record_id','title','description','source_url','retrieved_at']).issubset(df.columns)
print('Formatting and schema checks passed.')

## 3. Cleaning and quality checks

The cleaning policy keeps source wording intact in `description`, trims whitespace, preserves missing values as blank strings, and does not invent financial amounts. Where the council site reports a number, it is stored in `value` with an explicit `unit`.

In [ ]:
for col in df.columns:
    df[col] = df[col].astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()

quality = {
    'duplicate_record_ids': int(df['record_id'].duplicated().sum()),
    'missing_source_urls': int((df['source_url'] == '').sum()),
    'missing_titles': int((df['title'] == '').sum()),
    'records': len(df),
}
quality

In [ ]:
# Useful assignment-oriented views
facts = df[df.record_type == 'fact'][['record_id','title','value','unit','source_url']]
resources = df[df.record_type == 'resource'][['record_id','title','value','source_url']]
display(facts)
display(resources)

## 4. Export

The final export remains pipe-delimited and retains the required filename. The scraper in `scrape_kafue.py` can be rerun to refresh the page and resource inventory.

In [ ]:
OUTPUT = Path('db-unza26-csc4792-kafue_council_information.csv')
df.to_csv(OUTPUT, sep='|', index=False, encoding='utf-8')
print(f'Wrote {len(df)} records to {OUTPUT}')

## 5. Limitations and next enrichment pass

The council publications page links to detailed PDFs, XLSX, XLS, and DOCX files containing budgets, CDF submissions, expenditure returns, financial statements, project lists, performance reports, and minutes. This release records those official resources and their URLs. A stronger second release should download those files, extract tables, standardise ward/project names, and add monetary columns with explicit currency and financial year fields.